# Data centre — wind + grid operation electricity, at a timescale of your choosing

Runs the data centre's **Operation** electricity (currently modelled on the
ecoinvent US-SERC grid market — see
[2.tech_lca_foreground.ipynb](2.tech_lca_foreground.ipynb)) against a wind
installation you pick a location for (Renewables.ninja) blended with the GB
grid, over any date window you choose — reusing the same wind+grid machinery
already built for the electrolyser in [4.wind_power.ipynb](4.wind_power.ipynb).

**What this answers:** "if the data centre's operation electricity came from
my chosen wind site + grid top-up instead of the paper's US-SERC baseline,
what would emissions be for running it over *this* period?" — not the fixed
25-year total, but a total for whatever window `GRID_RANGE_START`/
`GRID_RANGE_END` (or `GRID_SINGLE_DATETIME`) selects in dashboard_config.py.

**How the 25-year functional unit gets massaged into a timescale:** the data
centre's foreground activity has one Operation electricity exchange — 1.60×10⁹
kWh over its 25-year life. This notebook reads that figure live from
Brightway (not re-typed, so it can't drift out of sync with notebook 2),
divides by 25 years (219,000 hours) to get an **average continuous power
draw** (~7.3 MW), then for every half-hour slice in your chosen window
multiplies `average power × slice length` to get the kWh actually drawn in
that slice, and prices that kWh at the blended wind/grid carbon intensity for
that same slice. Summing across the window gives total operation emissions
for exactly the period you asked for. Materials/embodied emissions are a
fixed one-off (not time-scalable) and are reported separately for context,
not folded into the window total.

**Scope note:** only `GRID_TIME_MODE = 'single'` or `'range'` are supported —
`'year_average'` represents a year via a handful of weighted representative
days, and turning that into a correct *total* (not just an average
intensity) needs an annualisation weight this notebook doesn't compute yet.
A real date range in `'range'` mode already *is* "a timescale of your
choosing" — a day, a week, a year, whatever `GRID_RANGE_START`/
`GRID_RANGE_END` covers in the underlying grid CSV.

Toggle with `RUN_DC_WIND_GRID_LCA`; choose `DC_WIND_LCA_MODE`
(`'blended'`/`'switching'`/`'both'`). Only the "cheap" scoring method is
supported (`DC_WIND_METHOD_MODE` is fixed to `'cheap'`) — the exact per-slice
Brightway LCA the electrolyser notebook also offers would mean one full LCA
solve per half-hour slice across the whole window, which doesn't scale the
way it does for a single kg of H2.

In [ ]:
# All settings come from dashboard_config.py — the single master dashboard.
# Edit values there once; every notebook picks them up.
from dashboard_config import *
import dashboard_config as cfg
import lca_helpers as H

print_dashboard()
print()
ei, bio, fg_db, method = H.setup_brightway()

In [ ]:
import bw2data as bd

if not RUN_DC_WIND_GRID_LCA:
    raise SystemExit(
        "RUN_DC_WIND_GRID_LCA is False in dashboard_config.py. Set it True and re-run."
    )

if DC_FOREGROUND_DB not in bd.databases:
    raise SystemExit(
        f"{DC_FOREGROUND_DB!r} not found. Run 2.tech_lca_foreground.ipynb's "
        "write cell first (material-production + operation-only data centre inventory)."
    )

print("Data centre wind/grid run")
print("-------------------------")
print("LCA mode:                  ", DC_WIND_LCA_MODE)
print("Grid time mode:            ", GRID_TIME_MODE)
if GRID_TIME_MODE == "single":
    print("Window:                    ", GRID_SINGLE_DATETIME)
else:
    print("Window:                    ", GRID_RANGE_START, "->", GRID_RANGE_END)
print("Wind installed cap. (kW):  ", WIND_INSTALLED_CAPACITY_KW)
print("Wind site (lat, lon):      ", WIND_LAT, WIND_LON)

## Resolve the data centre activity and its average power draw

In [ ]:
import bw2data as bd

dc_act = bd.get_activity((DC_FOREGROUND_DB, DC_CODE))
print("Data centre activity:", dc_act)

# Fixed (materials + any non-electricity operation inputs) score, and the
# total electricity found on the activity — the real, already-vetted 25-year
# SERC baseline total (1.60e9 kWh), read live from Brightway rather than
# re-typed, so it can't drift out of sync with notebook 3.
non_elec_score, DC_TOTAL_ELECTRICITY_KWH, _n_elec_exc = H.run_non_electricity_lca(dc_act, method)
print(f"Fixed (non-electricity) score, whole facility, 25 years: {non_elec_score:,.0f} kg CO2e")
print(f"Total baseline (SERC) operation electricity, 25 years:   {DC_TOTAL_ELECTRICITY_KWH:,.0f} kWh")

if _n_elec_exc != 1:
    raise ValueError(
        f"Expected exactly 1 direct electricity exchange on the data centre activity, "
        f"found {_n_elec_exc}. Check notebook 3's technosphere_exchanges_DC."
    )

DC_LIFETIME_HOURS    = 25 * 8760
DC_AVERAGE_POWER_KW  = DC_TOTAL_ELECTRICITY_KWH / DC_LIFETIME_HOURS
DC_MIN_WIND_POWER_KW = DC_AVERAGE_POWER_KW * DC_MIN_LOAD_FRACTION
print(f"\nImplied average continuous power draw: {DC_AVERAGE_POWER_KW:,.1f} kW "
      f"({DC_AVERAGE_POWER_KW/1000:,.2f} MW) over a {DC_LIFETIME_HOURS:,}-hour (25y) life.")
print(f"Switching threshold ({DC_MIN_LOAD_FRACTION:.0%} of average): {DC_MIN_WIND_POWER_KW:,.1f} kW")

## Load CSV and pick the timescale rows

In [ ]:
import pandas as pd

if GRID_TIME_MODE == "year_average":
    raise SystemExit(
        "GRID_TIME_MODE='year_average' isn't supported by this notebook (see intro markdown "
        "above) — set GRID_TIME_MODE to 'single' or 'range' in dashboard_config.py."
    )

selected_grid_source = getattr(cfg, "GRID_SOURCE_NORMALIZED", cfg.normalise_grid_data_source(getattr(cfg, "GRID_DATA_SOURCE", "csv")))

if selected_grid_source == "csv":
    # Use the old CSV selector, matching 3.custom_grid.ipynb.
    grid_df = H.load_grid_csv()
    run_rows, target_label = H.select_grid_rows(grid_df)
else:
    # For 4.1, reuse the exact API-derived rows selected by the grid notebook.
    if "DASHBOARD_GRID_RUN_ROWS" not in globals():
        raise RuntimeError(
            "GRID_DATA_SOURCE='carbon_api' requires the API-derived grid rows from "
            "3.1.custom_grid_carbon_intensity_api.ipynb. Run 1.dashboard_lca_adaptive.ipynb "
            "with RUN_GRID_NOTEBOOK_FROM_DASHBOARD=True, or run 4.1 first in this kernel."
        )
    run_rows = DASHBOARD_GRID_RUN_ROWS.copy()
    target_label = DASHBOARD_GRID_TARGET_LABEL

run_rows["DATETIME"] = pd.to_datetime(run_rows["DATETIME"]).dt.tz_localize(None)

print(f"Selected rows: {len(run_rows)}  ({target_label})")
print("Grid data source:", selected_grid_source)
print("Start:", run_rows["DATETIME"].min())
print("End  :", run_rows["DATETIME"].max())
_preview_cols = [c for c in ["DATETIME", "GENERATION", "CARBON_INTENSITY", "WIND", "WIND_perc"] if c in run_rows.columns]
run_rows[_preview_cols].head()

## Renewables.ninja session + wind data fetch

In [ ]:
date_from = run_rows["DATETIME"].min().date().isoformat()
date_to   = run_rows["DATETIME"].max().date().isoformat()

def _fetch_with_session(session):
    return H.fetch_ninja_wind(
        session=session,
        lat=WIND_LAT, lon=WIND_LON,
        date_from=date_from, date_to=date_to,
        capacity_kw=WIND_INSTALLED_CAPACITY_KW,
        height_m=NINJA_HUB_HEIGHT_M,
        turbine=NINJA_TURBINE,
        dataset=NINJA_DATASET,
    )

# Try current token (env/prompt), then allow up to 2 explicit re-entry retries.
max_attempts = 3
last_exc = None
for attempt in range(1, max_attempts + 1):
    if attempt == 1:
        ninja_session = H.create_ninja_session()
    else:
        from getpass import getpass
        prompt_fn = getpass if USE_GETPASS else input
        retry_token = prompt_fn(
            f"Re-enter Renewables.ninja API token (attempt {attempt}/{max_attempts}, blank to abort): "
        ).strip()
        if not retry_token:
            raise RuntimeError("Wind data fetch aborted: no retry token provided.") from last_exc
        ninja_session = H.create_ninja_session(token=retry_token)

    try:
        wind_hourly_df, ninja_metadata, ninja_args = _fetch_with_session(ninja_session)
        break
    except RuntimeError as exc:
        last_exc = exc
        msg = str(exc).lower()
        if "authentication failed" not in msg and "status 401" not in msg and "status 403" not in msg:
            raise
        print("\nRenewables.ninja auth failed.")
        print("Check token validity and account quota, then retry.")
else:
    raise RuntimeError(
        "Wind data fetch failed after multiple token attempts. "
        "Set a valid RENEWABLES_NINJA_TOKEN and rerun this cell."
    ) from last_exc

if wind_hourly_df is None or wind_hourly_df.empty:
    raise RuntimeError("Renewables.ninja returned no wind data for the selected dates.")

wind_hourly_df.head()

## Align wind power onto half-hour slices (against the data centre's average demand)

In [ ]:
aligned_rows = H.align_wind_to_grid_timeslices(
    wind_hourly_df, run_rows, method=WIND_TO_HALFHOUR_METHOD,
    demand_kw=DC_AVERAGE_POWER_KW, min_wind_power_kw=DC_MIN_WIND_POWER_KW,
)

_preview_mode = "switching" if DC_WIND_LCA_MODE == "switching" else "blended"
if _preview_mode == "blended":
    print("Mean wind fraction:", aligned_rows["wind_fraction"].mean())
    print("Mean grid fraction:", aligned_rows["grid_fraction"].mean())
    cols = ["DATETIME", "wind_power_kw", "wind_fraction", "grid_fraction",
            "wind_used_kw", "grid_topup_kw", "surplus_wind_kw", "CARBON_INTENSITY"]
else:
    print("Slices per source:")
    print(aligned_rows["electricity_source"].value_counts())
    cols = ["DATETIME", "wind_power_kw", "minimum_required_wind_kw",
            "wind_above_minimum", "electricity_source", "CARBON_INTENSITY"]
aligned_rows[cols].head(12)

## Pick the wind electricity background process & build foreground process

In [ ]:
wind_candidates = H.show_candidates(WIND_BACKGROUND_QUERY, max_results=20)
if not wind_candidates:
    raise ValueError("No wind electricity candidates found. Check WIND_BACKGROUND_QUERY.")

wind_background_act = wind_candidates[WIND_BACKGROUND_INDEX]
print("\nSelected wind background process:")
print(wind_background_act)

wind_electricity_act = H.build_wind_electricity_activity(wind_background_act, fg_db)
print("\nCreated foreground wind electricity process:")
print(wind_electricity_act)

## Per-kWh electricity scores (wind + grid), cheap method

In [ ]:
# Grid template dicts are defined in 3.custom_grid.ipynb.
for _req in ("GRID_CANDIDATE_INDEX", "GRID_INFRASTRUCTURE", "GRID_EMISSIONS"):
    if _req not in globals():
        raise NameError(
            f"{_req} is not defined. Run 3.custom_grid.ipynb first "
            "(or run the dashboard with RUN_GRID_NOTEBOOK=True)."
        )

print("Pre-computing one-off LCA scores...")
wind_elec_score = H.run_lca_score(wind_electricity_act, method)
print(f"  wind electricity score: {wind_elec_score:.6f} kg CO2e/kWh")

_gen = H.select_all_candidate_activities(GRID_CANDIDATE_INDEX)
grid_source_scores = H.calculate_source_lca_scores(_gen, method)

## Run the emissions accumulation across the chosen window

In [ ]:
import numpy as np

# Slice duration read from the actual timestamps rather than hardcoded, so
# this still works if the underlying grid CSV resolution ever changes.
_dt_diffs = aligned_rows["DATETIME"].diff().dropna().dt.total_seconds() / 3600.0
SLICE_HOURS = float(_dt_diffs.median()) if len(_dt_diffs) else 0.5
print(f"Slice duration: {SLICE_HOURS:.4f} h  |  {len(aligned_rows)} slice(s) in the window")

_effective_modes = ["blended", "switching"] if DC_WIND_LCA_MODE == "both" else [DC_WIND_LCA_MODE]
_all_records = []

for _eff_mode in _effective_modes:
    for _, r in aligned_rows.iterrows():
        kwh = DC_AVERAGE_POWER_KW * SLICE_HOURS
        grid_score, _ = H.custom_electricity_score_for_row(r, grid_source_scores)

        if _eff_mode == "blended":
            wf = float(r["wind_fraction"]); gf = float(r["grid_fraction"])
            intensity = wf * wind_elec_score + gf * grid_score
        else:
            wf = 1.0 if r["electricity_source"] == "wind" else 0.0
            gf = 1.0 - wf
            intensity = wind_elec_score if wf == 1.0 else grid_score

        _all_records.append({
            "datetime": r["DATETIME"], "lca_mode": _eff_mode,
            "wind_power_kw": r.get("wind_power_kw", np.nan),
            "wind_fraction": wf, "grid_fraction": gf,
            "kwh_this_slice": kwh,
            "grid_carbon_intensity_kgco2_per_kwh": grid_score,
            "wind_carbon_intensity_kgco2_per_kwh": wind_elec_score,
            "blended_intensity_kgco2_per_kwh": intensity,
            "emissions_kgco2e_this_slice": kwh * intensity,
        })

slice_df = pd.DataFrame(_all_records).sort_values(["lca_mode", "datetime"]).reset_index(drop=True)
print(f"\nComputed {len(slice_df)} slice-mode row(s).")
slice_df.head()

## Totals for the chosen window

In [ ]:
summary_rows = []
for _eff_mode, mode_df in slice_df.groupby("lca_mode"):
    total_kwh = mode_df["kwh_this_slice"].sum()
    total_emissions = mode_df["emissions_kgco2e_this_slice"].sum()
    window_hours = len(mode_df) * SLICE_HOURS
    summary_rows.append({
        "lca_mode": _eff_mode,
        "window_start": mode_df["datetime"].min(),
        "window_end": mode_df["datetime"].max(),
        "window_hours": window_hours,
        "window_days": window_hours / 24.0,
        "total_kwh": total_kwh,
        "total_operation_emissions_kgco2e": total_emissions,
        "effective_avg_intensity_kgco2_per_kwh": total_emissions / total_kwh if total_kwh else np.nan,
        "mean_wind_fraction": mode_df["wind_fraction"].mean(),
    })
summary_df = pd.DataFrame(summary_rows).set_index("lca_mode")

print(f"Fixed materials/non-electricity score (whole 25-year facility, one-off): "
      f"{non_elec_score:,.0f} kg CO2e")
print("(not scaled by the window below — embodied burden isn't a rate)")
print()
summary_df

## Plot

In [ ]:
import matplotlib.pyplot as plt

_plot_modes = ["blended", "switching"] if DC_WIND_LCA_MODE == "both" else [DC_WIND_LCA_MODE]

for _pm in _plot_modes:
    pm_df = slice_df[slice_df["lca_mode"] == _pm].sort_values("datetime")
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

    axes[0].plot(pm_df["datetime"], pm_df["blended_intensity_kgco2_per_kwh"] * 1000,
                 color="#2c3e50", lw=1.2)
    axes[0].set_ylabel("Carbon intensity\n(g CO2e/kWh)")
    axes[0].set_title(f"Data centre operation electricity — {_pm} mode, "
                       f"{pm_df['datetime'].min()} to {pm_df['datetime'].max()}")

    axes[1].plot(pm_df["datetime"], pm_df["emissions_kgco2e_this_slice"].cumsum() / 1000.0,
                 color="#c0392b", lw=1.5)
    axes[1].set_ylabel("Cumulative emissions\n(tonnes CO2e)")
    axes[1].set_xlabel("Date/time")

    plt.tight_layout()
    plt.show()

## Export results CSV

In [ ]:
from pathlib import Path

output_dir = Path("datacentre_wind_grid_lca_outputs")
output_dir.mkdir(exist_ok=True)
safe_window = str(target_label).replace(" ", "T").replace(":", "-").replace("->", "_to_")

slice_csv = output_dir / f"dc_wind_grid_slices_{safe_window}.csv"
slice_df.to_csv(slice_csv, index=False)

summary_csv = output_dir / f"dc_wind_grid_summary_{safe_window}.csv"
summary_df.to_csv(summary_csv)

print("Saved per-slice results to:", slice_csv.resolve())
print("Saved summary to:          ", summary_csv.resolve())